In [1]:
import os
import json
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset

# -------- Dataset --------
class RelativeSpeedDataset200D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"):
                continue
            sid = fname.replace(".json", "")
            if sid not in self.distances:
                continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            if len(seq) < 20:
                continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            if len(keys) < 20:
                continue
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)

            def smooth(x, w):
                return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items:
                    return

                d = dist[i:i+20]
                o = own[i:i+20]
                t = tgt[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)):
                    continue

                rel_speed = t - o
                own_acc = np.gradient(o)
                d1 = np.gradient(d)
                d2 = np.gradient(d1)

                f3 = smooth(d, 3)
                f5 = smooth(d, 5)
                f7 = smooth(d, 7)
                f11 = smooth(d, 11)
                f11_d1 = np.gradient(f11)

                try:
                    feat = np.concatenate([
                        d, o, own_acc, d1, d2,
                        f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20]
                    ])
                except:
                    continue

                if feat.shape[0] != 200:
                    continue

                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target, sid))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, tgt, sid = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32), sid

# -------- モデル定義（高性能構成） --------
class EnhancedLSTMModel(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=256, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.Softplus(),  # 滑らかな活性化
            nn.Linear(128, 64),
            nn.Softplus(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.view(x.size(0), 20, 10)
        out, _ = self.lstm(x)        # (B, 20, hidden_dim)
        pooled = out.mean(dim=1)     # 平均プーリング
        return self.fc(pooled).squeeze(1)

# -------- 学習ループ --------
def train_enhanced_lstm(dataset, save_path="model_enhanced_lstm.pth"):
    scenes = sorted(set([item[-1] for item in dataset.items]))
    train_scenes, val_scenes = train_test_split(scenes, test_size=0.2, random_state=42)
    train_idx = [i for i, item in enumerate(dataset.items) if item[-1] in train_scenes]
    val_idx = [i for i, item in enumerate(dataset.items) if item[-1] in val_scenes]

    train_ds = Subset(dataset, train_idx)
    val_ds = Subset(dataset, val_idx)

    def collate_fn(batch):
        feats, tgts, _ = zip(*batch)
        return torch.stack(feats), torch.tensor(tgts)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = EnhancedLSTMModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.SmoothL1Loss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_loss = float('inf')
    patience = 30
    min_delta = 0.0003
    counter = 0

    for epoch in range(100):
        model.train()
        total_train_loss = 0
        for feats, tgts in tqdm(train_loader, desc=f"[Train {epoch+1}]"):
            feats, tgts = feats.to(device), tgts.to(device)
            optimizer.zero_grad()
            loss = criterion(model(feats), tgts)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * feats.size(0)

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for feats, tgts in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                loss = criterion(model(feats), tgts)
                total_val_loss += loss.item() * feats.size(0)

        train_loss = total_train_loss / len(train_ds)
        val_loss = total_val_loss / len(val_ds)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        if best_val_loss - val_loss > min_delta:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"✅ Saved model to {save_path} (val_loss={val_loss:.4f})")
            counter = 0
        else:
            counter += 1
            print(f"⏸ No improvement. Patience: {counter}/{patience}")
            if counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break

    return model

# -------- 実行部 --------
if __name__ == "__main__":
    dataset = RelativeSpeedDataset200D(
        annot_root="../train/train_annotations",
        distance_json_path="../train2/distance1/corrected_distance_estimates_filtered.json",
        max_items=7500
    )

    model = train_enhanced_lstm(dataset, save_path="model_enhanced_lstm.pth")
    print("✅ 学習完了: model_enhanced_lstm.pth に保存しました")


[Train 1]: 100%|██████████| 92/92 [00:00<00:00, 105.23it/s]


Epoch 1 | Train Loss: 2.8044 | Val Loss: 0.6800
✅ Saved model to model_enhanced_lstm.pth (val_loss=0.6800)


[Train 2]: 100%|██████████| 92/92 [00:00<00:00, 181.01it/s]


Epoch 2 | Train Loss: 0.2566 | Val Loss: 0.1362
✅ Saved model to model_enhanced_lstm.pth (val_loss=0.1362)


[Train 3]: 100%|██████████| 92/92 [00:00<00:00, 180.15it/s]


...

Epoch 71 | Train Loss: 0.0322 | Val Loss: 0.0460
⏸ No improvement. Patience: 28/30


[Train 72]: 100%|██████████| 92/92 [00:00<00:00, 175.67it/s]


Epoch 72 | Train Loss: 0.0322 | Val Loss: 0.0460
⏸ No improvement. Patience: 29/30


[Train 73]: 100%|██████████| 92/92 [00:00<00:00, 180.18it/s]


Epoch 73 | Train Loss: 0.0322 | Val Loss: 0.0451
⏸ No improvement. Patience: 30/30
🛑 Early stopping at epoch 73
✅ 学習完了: model_enhanced_lstm.pth に保存しました
